# Tahap 2: Preprocessing & Feature Engineering Dasar
Notebook ini didedikasikan untuk mengeksekusi rencana dari hasil temuan EDA, membersihkan data, dan menyiapkan subset validasi lokal (Time-Series Split).

## 1. Import Library & Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

## 2. Load Data

In [2]:
train = pd.read_csv('../dataset/train.csv')
test = pd.read_csv('../dataset/test.csv')
test_history = pd.read_csv('../dataset/test_history.csv')
movies = pd.read_csv('../dataset/movies.csv')
holidays = pd.read_csv('../dataset/holidays.csv')
ticket_prices = pd.read_csv('../dataset/ticket_prices.csv')
city_coordinates = pd.read_csv('../dataset/city_coordinates.csv')
weather_data = pd.read_csv('../dataset/weather_data.csv')

print('Semua data berhasil dimuat.')

Semua data berhasil dimuat.


## 3. Data Cleansing (Penghapusan Anomali)

In [3]:
# C. Pembersihan Ekstra (Berdasarkan Full-Scale Validation M5)
import re

# 1. Menghapus Suffix Versi pada Judul Film (train, test_history, test)
# Contoh: "PABRIK GULA (IMAX 2D)" -> "PABRIK GULA"
def clean_movie_title(title):
    if not isinstance(title, str): return title
    # Hapus suffix seperti (IMAX 2D), (3D), UNCUT
    title = re.sub(r'\s*\(.*?\)', '', title)
    title = title.replace(' UNCUT', '')
    return title.strip()

for df in [train, test_history, test]:
    if 'movie_title' in df.columns:
        df['movie_title'] = df['movie_title'].apply(clean_movie_title)

# 2. Injeksi Koordinat untuk Kota Yatim di ticket_prices.csv
missing_coords = pd.DataFrame([
    {'city_name': 'MAGELANG', 'lat': -7.4812, 'lon': 110.2173},
    {'city_name': 'TUBAN', 'lat': -6.8979, 'lon': 112.0649}
])
city_coordinates = pd.concat([city_coordinates, missing_coords], ignore_index=True)
city_coordinates.drop_duplicates(subset=['city_name'], inplace=True)

# 3. Standarisasi Nama Libur (holidays.csv)
holidays.loc[holidays['holiday_name'] == 'Kelahiran Yesus Kristus', 'holiday_name'] = 'Hari Raya Natal'

# 4. Imputasi Missing Values pada Fitur Baru movies.csv (runtime_minutes & director_popularity)
movies['runtime_minutes'] = movies['runtime_minutes'].fillna(movies['runtime_minutes'].median())
movies['director_popularity'] = movies['director_popularity'].fillna(0.0)

print("Pembersihan anomali (Suffix film, Koordinat, Hari Libur, Imputasi Movies) selesai!")


Pembersihan anomali (Suffix film, Koordinat, Hari Libur, Imputasi Movies) selesai!


In [4]:
print(f'Ukuran Train Awal: {train.shape}')

# A. Hapus Duplikasi
train.drop_duplicates(inplace=True)
test_history.drop_duplicates(inplace=True)

# B. Hapus Anomali Semantik & Multivariat di Train
train = train[(train['total_ticket'] >= 0) & 
              (train['occupation_rate'] >= 0) & (train['occupation_rate'] <= 100) &
              (train['total_show'] > 0)]

# C. Hapus Kapasitas Mustahil (>1000 per show)
train = train[(train['total_ticket'] / train['total_show']) <= 1000]

# D. Bersihkan Metadata Film (Ganti '-' dengan 'Unknown')
for col in ['producer', 'director', 'writer', 'casts']:
    movies[col] = movies[col].replace('-', 'Unknown').fillna('Unknown')

# E. Bersihkan Tabel Hari Libur
holidays['holiday_name'] = holidays['holiday_name'].fillna('Unknown Holiday')

print(f'Ukuran Train Setelah Cleansing: {train.shape}')

Ukuran Train Awal: (138959, 11)
Ukuran Train Setelah Cleansing: (138959, 11)


## 4. Optimalisasi Tipe Data (Memory Downcasting)

In [5]:
def optimize_datatypes(df):
    # 1. Parsing Datetime
    for col in ['date_show', 'date']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
            
    # 2. Category Casting
    for col in ['age_rating', 'day_tipe', 'holiday_tipe', 'price_day']:
        if col in df.columns:
            df[col] = df[col].astype('category')
            
    # 3. Integer Downcasting
    for col in ['total_ticket', 'total_show']:
        if col in df.columns:
            df[col] = df[col].astype('int32')
            
    return df

train = optimize_datatypes(train)
test_history = optimize_datatypes(test_history)
test = optimize_datatypes(test)
movies = optimize_datatypes(movies)
holidays = optimize_datatypes(holidays)
ticket_prices = optimize_datatypes(ticket_prices)

print('Konversi memori (datetime & category) berhasil diaplikasikan.')

Konversi memori (datetime & category) berhasil diaplikasikan.


## 5. Integrasi Relasional (Left Join)

In [6]:
def build_features(df):
    df_feat = df.copy()
    
    # A. Ekstraksi Komponen Waktu Lokal (Untuk Join Ticket Prices)
    df_feat['day_of_week'] = df_feat['date_show'].dt.dayofweek
    df_feat['extracted_day_tipe'] = np.where(df_feat['day_of_week'] < 4, 'Weekday', 
                                    np.where(df_feat['day_of_week'] == 4, 'Friday', 'Weekend'))
    
    # B. Ekstraksi Sifat Kalender Lanjutan
    df_feat['month'] = df_feat['date_show'].dt.month
    df_feat['is_weekend'] = df_feat['day_of_week'].isin([4, 5, 6]).astype('int8')
    df_feat['is_month_end'] = df_feat['date_show'].dt.is_month_end.astype('int8')
    
    # C. Join Data Movies (Metadata Film)
    df_feat = df_feat.merge(movies, left_on='movie_title', right_on='original_title', how='left')
    
    # D. Penanganan Orphan Data (Film tanpa profil di movies.csv)
    df_feat['genre'] = df_feat['genre'].fillna('Unknown')
    if 'Unknown' not in df_feat['age_rating'].cat.categories:
        df_feat['age_rating'] = df_feat['age_rating'].cat.add_categories('Unknown')
    df_feat['age_rating'] = df_feat['age_rating'].fillna('Unknown')
    
    # E. Join Hari Libur
    df_feat = df_feat.merge(holidays, left_on='date_show', right_on='date', how='left')
    if 'normal' not in df_feat['holiday_tipe'].cat.categories:
        df_feat['holiday_tipe'] = df_feat['holiday_tipe'].cat.add_categories('normal')
    df_feat['holiday_tipe'] = df_feat['holiday_tipe'].fillna('normal')
    df_feat['holiday_name'] = df_feat['holiday_name'].fillna('Not a Holiday')
    
    # F. Join Harga Tiket Dasar
    df_feat = df_feat.merge(ticket_prices, left_on=['city_name', 'extracted_day_tipe'], 
                            right_on=['city_name', 'price_day'], how='left')
    
    # Bersihkan sisa kolom redundan
    cols_to_drop = ['original_title', 'date', 'price_day', 'extracted_day_tipe']
    df_feat.drop(columns=[c for c in cols_to_drop if c in df_feat.columns], inplace=True, errors='ignore')
    
    return df_feat

train_full = build_features(train)
print(f'Bentuk data train setelah seluruh Feature Engineering Dasar: {train_full.shape}')

Bentuk data train setelah seluruh Feature Engineering Dasar: (138959, 27)


## 6. Target Transformation & Validasi Waktu (Local CV)

In [7]:
# Terapkan log1p pada target untuk meredam skewness ekstrem
train_full['log_total_ticket'] = np.log1p(train_full['total_ticket'])

# Time-Series Split: Hold-out bulan terakhir (September 2025) sebagai Evaluasi Lokal
train_set = train_full[train_full['date_show'] < '2025-09-01'].copy()
val_set = train_full[train_full['date_show'] >= '2025-09-01'].copy()

print(f'Ukuran Training Set (April - Agustus): {train_set.shape}')
print(f'Ukuran Local Validation Set (September): {val_set.shape}')

# Ekspor Dataset
train_full.to_parquet('../dataset/cleaned_dataset.parquet')
print("Data berhasil diekspor sebagai '../dataset/cleaned_dataset.parquet'!")
print('\nData telah disterilkan dan terintegrasi! Siap dilanjutkan ke Model Eksperimen.')

Ukuran Training Set (April - Agustus): (115610, 28)
Ukuran Local Validation Set (September): (23349, 28)
Data berhasil diekspor sebagai '../dataset/cleaned_dataset.parquet'!

Data telah disterilkan dan terintegrasi! Siap dilanjutkan ke Model Eksperimen.
